### **COLAB File Path**

In [ ]:
import os
import json
from datetime import datetime

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab, skipping Drive mount.")

BASE_DIR = "/content/drive/MyDrive/547 Final Project/agents"
os.makedirs(BASE_DIR, exist_ok=True)

INTAKE_PATH = os.path.join(BASE_DIR, "initial_output.json")
PLANNER_PATH = os.path.join(BASE_DIR, "planner_payload.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Initial Def**

In [ ]:
def ask_text(prompt: str, default: str = "") -> str:
    suffix = f" [{default}]" if default else ""
    value = input(f"{prompt}{suffix}: ").strip()
    return value if value else default


def ask_int(prompt: str, default: int = 3, min_value: int = 1, max_value: int = 365) -> int:
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        if not raw:
            return default
        try:
            value = int(raw)
            if not (min_value <= value <= max_value):
                print(f"Please enter a number between {min_value} and {max_value}.")
                continue
            return value
        except ValueError:
            print("Please enter a valid integer.")


def ask_date(prompt: str) -> str:
    while True:
        raw = input(f"{prompt} (YYYY-MM-DD, leave blank if unknown): ").strip()
        if not raw:
            return ""
        try:
            datetime.strptime(raw, "%Y-%m-%d")
            return raw
        except ValueError:
            print("Invalid date format. Please use YYYY-MM-DD.")


def ask_yes_no(prompt: str, default: bool = False) -> bool:
    default_text = "y" if default else "n"
    raw = input(f"{prompt} [y/n, default {default_text}]: ").strip().lower()
    if not raw:
        return default
    return raw in ("y", "yes", "true", "1")


def ask_single_choice(prompt: str, options: list[str], default_index: int = 0) -> str:
    print(f"\n{prompt}")
    for i, option in enumerate(options, start=1):
        suffix = " (default)" if i - 1 == default_index else ""
        print(f"  {i}. {option}{suffix}")

    while True:
        raw = input(f"Choose 1-{len(options)} [{default_index + 1}]: ").strip()
        if not raw:
            return options[default_index]
        try:
            idx = int(raw)
            if 1 <= idx <= len(options):
                return options[idx - 1]
            print("Choice out of range.")
        except ValueError:
            print("Please enter a number.")


def ask_multi_choice(prompt: str, options: list[str]) -> list[str]:
    print(f"\n{prompt}")
    for i, option in enumerate(options, start=1):
        print(f"  {i}. {option}")
    print("Enter numbers separated by commas, or press Enter for none.")

    while True:
        raw = input("Your choices: ").strip()
        if not raw:
            return []
        try:
            indexes = [int(x.strip()) for x in raw.split(",") if x.strip()]
            selected = []
            for idx in indexes:
                if 1 <= idx <= len(options):
                    item = options[idx - 1]
                    if item not in selected:
                        selected.append(item)
                else:
                    raise ValueError
            return selected
        except ValueError:
            print("Invalid input. Example: 1,3,5")


def save_json(path: str, payload: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Saved: {path}")


# Standardized input schema

ACTIVITY_OPTIONS = [
    "beaches",
    "city_sightseeing",
    "outdoor_adventures",
    "festivals_events",
    "food_exploration",
    "nightlife",
    "shopping",
    "spa_wellness",
]

BUDGET_OPTIONS = ["low", "medium", "high"]
COMPANION_OPTIONS = ["solo", "couple", "family", "friends"]


def collect_trip_intake() -> dict:
    print("=== Tell us your travel preferences ===")

    destination = ask_text("What is destination of choice?")
    travel_date = ask_date("When are you planning to travel?")
    days = ask_int("How many days are you planning to travel?", default=5)

    budget_level = ask_single_choice("What is your budget?", BUDGET_OPTIONS, default_index=1)
    companions = ask_single_choice("Who do you plan on traveling with?", COMPANION_OPTIONS, default_index=0)
    activities = ask_multi_choice("Which activities are you interested in?", ACTIVITY_OPTIONS)

    print("\nAny food preferences?")
    halal = ask_yes_no("Halal", default=False)
    vegan = ask_yes_no("Vegan", default=False)

    extra_preferences = ask_text("Any extra preferences or constraints? (optional)", "")

    budget_ranges = {
        "low (0 - 1000 USD)": {"min_usd": 0, "max_usd": 1000},
        "medium (1000 - 2500 USD)": {"min_usd": 1000, "max_usd": 2500},
        "high (2500 + USD)": {"min_usd": 2500, "max_usd": None},
    }

    intake = {
        "schema_version": "1.0",
        "agent_type": "travel_intake",
        "destination": destination,
        "travel_date": travel_date,
        "trip_duration_days": days,
        "budget": {
            "level": budget_level,
            "currency": "USD",
            "range": budget_ranges[budget_level],
        },
        "travel_companions": companions,
        "activity_preferences": activities,
        "food_preferences": {
            "halal": halal,
            "vegan": vegan,
        },
        "extra_preferences": extra_preferences,
        "created_at": datetime.utcnow().isoformat() + "Z",
    }

    return intake


def build_planner_payload(intake: dict) -> dict:
    activities = intake.get("activity_preferences", [])
    food = intake.get("food_preferences", {})

    preferences = []
    preferences.append(f"budget:{intake.get('budget', {}).get('level', 'unknown')}")
    preferences.append(f"companions:{intake.get('travel_companions', 'unknown')}")
    preferences.extend([f"activity:{x}" for x in activities])

    if food.get("halal"):
        preferences.append("food:halal")
    if food.get("vegan"):
        preferences.append("food:vegan")

    if intake.get("extra_preferences"):
        preferences.append(f"note:{intake['extra_preferences']}")

    research_notes = {
        "destination": intake["destination"],
        "recommended_areas": [],
        "attractions": [],
        "planning_hints": [
            f"Travel date: {intake['travel_date'] or 'unspecified'}",
            f"Budget level: {intake['budget']['level']}",
            f"Travel companions: {intake['travel_companions']}",
            f"Activity preferences: {', '.join(activities) if activities else 'none'}",
            f"Food preferences: halal={food.get('halal', False)}, vegan={food.get('vegan', False)}",
            f"Extra preferences: {intake['extra_preferences'] or 'none'}",
        ],
        "constraints": [
            f"Trip duration: {intake['trip_duration_days']} days",
            "Use a realistic pace and group activities by area.",
        ],
    }

    return {
        "destination": intake["destination"],
        "days": intake["trip_duration_days"],
        "preferences": preferences,
        "research_notes": research_notes,
    }

### **Input def**

In [15]:
def main():
    intake = collect_trip_intake()
    planner_payload = build_planner_payload(intake)

    print("\n=== Standardized Intake ===")
    print(json.dumps(intake, ensure_ascii=False, indent=2))

    print("\n=== Planner Payload ===")
    print(json.dumps(planner_payload, ensure_ascii=False, indent=2))

    save_json(INTAKE_PATH, intake)
    save_json(PLANNER_PATH, planner_payload)

    print("\nDone.")


if __name__ == "__main__":
    main()

=== Travel Input Wizard ===
Destination [Tokyo]: alaska
When are you planning to travel? (YYYY-MM-DD, leave blank if unknown): 2026-06-07
How many days are you planning to travel? [3]: 5

What is your budget?
  1. low
  2. medium (default)
  3. high
Choose 1-3 [2]: 2

Who do you plan on traveling with?
  1. solo (default)
  2. couple
  3. family
  4. friends
Choose 1-4 [1]: 3

Which activities are you interested in?
  1. beaches
  2. city_sightseeing
  3. outdoor_adventures
  4. festivals_events
  5. food_exploration
  6. nightlife
  7. shopping
  8. spa_wellness
Enter numbers separated by commas, or press Enter for none.
Your choices: 1,3,5

Any food preferences?
Halal [y/n, default n]: y/n
Vegan [y/n, default n]: n
Any extra preferences or constraints? (optional): i'm homosexual.

=== Standardized Intake ===
{
  "schema_version": "1.0",
  "agent_type": "travel_intake",
  "destination": "alaska",
  "travel_date": "2026-06-07",
  "trip_duration_days": 5,
  "budget": {
    "level": "med

/tmp/ipykernel_6576/1106483346.py:150: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
